In [16]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
import math

In [17]:
# Only needed once
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
data_csv = pd.read_csv('995,000_rows.csv')
# extract 300,000 rows
data = data_csv.head(300000)
data.head()

C:\Users\bruge\AppData\Local\Temp\ipykernel_29300\2544411185.py:1: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  data_csv = pd.read_csv('995,000_rows.csv')


,Unnamed: 0,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary,source
0,732,7444726.0,nationalreview.com,political,http://www.nationalreview.com/node/152734/%E2%...,Plus one article on Google Plus\n\n(Thanks to ...,2017-11-27T01:14:42.983556,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,Iran News Round Up,NaN,NaN,"['National Review', 'National Review Online', ...",NaN,NaN,NaN,NaN
1,1348,6213642.0,beforeitsnews.com,fake,http://beforeitsnews.com/economy/2012/06/the-c...,The Cost Of The Best Senate Banking Committee ...,2017-11-27T01:14:08.7454,2018-02-08 19:18:34.468038,2018-02-08 19:18:34.468066,The Cost Of The Best Senate Banking Committee ...,NaN,NaN,[''],NaN,NaN,NaN,NaN
2,7119,3867639.0,dailycurrant.com,satire,http://dailycurrant.com/2016/01/18/man-awoken-...,Man Awoken From 27-Year Coma Commits Suicide A...,2017-11-27T01:14:21.395055,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,Man Awoken From 27-Year Coma Commits Suicide A...,NaN,NaN,[''],NaN,NaN,NaN,NaN
3,1518,9560791.0,nytimes.com,reliable,https://query.nytimes.com/gst/fullpage.html?re...,WHEN Julia Geist was asked to draw a picture o...,2018-02-11 00:46:42.632962,2018-02-11 00:14:20.346838,2018-02-11 00:14:20.346871,Opening a Gateway for Girls to Enter the Compu...,NaN,NaN,"['Computers and the Internet', 'Women and Girl...",WHEN Julia Geist was asked to draw a picture o...,NaN,NaN,nytimes
4,9345,2059625.0,infiniteunknown.net,conspiracy,http://www.infiniteunknown.net/2011/09/14/100-...,– 100 Compiled Studies on Vaccine Dangers (Act...,2017-11-10T11:18:44.524042,2018-02-07 23:39:33.852671,2018-02-07 23:39:33.852696,100 Compiled Studies on Vaccine Dangers – Infi...,NaN,NaN,[''],NaN,"Lymphoma, Hepatitis B, Immune System, Health, ...",NaN,NaN


## Clean the Data

In [19]:
def clean_text(text):
    # 1. Håndter tomme værdier (vigtigt for det store datasæt)
    if pd.isna(text):
        return ""
    
    # 2. Gør alt til små bogstaver
    text = text.lower()

    # 3. Erstat URLs med et tag (bevarer information om at der var et link)
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)

    # 4. Erstat e-mails med et tag
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '<EMAIL>', text)

    # 5. Erstat datoer (f.eks. 13th) med et tag
    text = re.sub(r'[0-9]+[a-zA-Z]+', '<DATE>', text)

    # 6. Erstat resterende tal med et tag
    text = re.sub(r'[0-9]+', '<NUM>', text)
    
    # 7. Fjern specialtegn (behold kun bogstaver, tags og mellemrum)
    text = re.sub(r'[^a-z\s<>]', '', text)
    
    # 8. Fjern ekstra mellemrum, tabs og linjeskift
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

data["clean_content"] = data["content"].apply(clean_text)
data[["content", "clean_content"]].head()

C:\Users\bruge\AppData\Local\Temp\ipykernel_29300\2796242259.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["clean_content"] = data["content"].apply(clean_text)


,content,clean_content
0,Plus one article on Google Plus\n\n(Thanks to ...,plus one article on google plus thanks to ali ...
1,The Cost Of The Best Senate Banking Committee ...,the cost of the best senate banking committee ...
2,Man Awoken From 27-Year Coma Commits Suicide A...,man awoken from <>year coma commits suicide af...
3,WHEN Julia Geist was asked to draw a picture o...,when julia geist was asked to draw a picture o...
4,– 100 Compiled Studies on Vaccine Dangers (Act...,<> compiled studies on vaccine dangers activis...


In [20]:
data['tokens'] = data['clean_content'].apply(nltk.word_tokenize)
data[['clean_content', 'tokens']].head()

C:\Users\bruge\AppData\Local\Temp\ipykernel_29300\2812601675.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['tokens'] = data['clean_content'].apply(nltk.word_tokenize)


,clean_content,tokens
0,plus one article on google plus thanks to ali ...,"[plus, one, article, on, google, plus, thanks,..."
1,the cost of the best senate banking committee ...,"[the, cost, of, the, best, senate, banking, co..."
2,man awoken from <>year coma commits suicide af...,"[man, awoken, from, <, >, year, coma, commits,..."
3,when julia geist was asked to draw a picture o...,"[when, julia, geist, was, asked, to, draw, a, ..."
4,<> compiled studies on vaccine dangers activis...,"[<, >, compiled, studies, on, vaccine, dangers..."


In [21]:
# Store English stopwords into a variable
english_stopwords = set(stopwords.words('english'))
print(f"Number of stopwords: {len(english_stopwords)}")
print(list(english_stopwords)[:15]) # Display first 15 stopwords

data['filtered_tokens'] = data['tokens'].apply(lambda tokens: [word for word in tokens if word not in english_stopwords])
data[['tokens', 'filtered_tokens']].head()

Number of stopwords: 198
["he'll", 'such', 'her', 'm', "isn't", "they're", 'hers', 'with', "weren't", 'having', "should've", 'nor', 'before', 'his', 'it']


C:\Users\bruge\AppData\Local\Temp\ipykernel_29300\1738622045.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['filtered_tokens'] = data['tokens'].apply(lambda tokens: [word for word in tokens if word not in english_stopwords])


,tokens,filtered_tokens
0,"[plus, one, article, on, google, plus, thanks,...","[plus, one, article, google, plus, thanks, ali..."
1,"[the, cost, of, the, best, senate, banking, co...","[cost, best, senate, banking, committee, jp, m..."
2,"[man, awoken, from, <, >, year, coma, commits,...","[man, awoken, <, >, year, coma, commits, suici..."
3,"[when, julia, geist, was, asked, to, draw, a, ...","[julia, geist, asked, draw, picture, computer,..."
4,"[<, >, compiled, studies, on, vaccine, dangers...","[<, >, compiled, studies, vaccine, dangers, ac..."


In [22]:
# Liste over alle ord i alle artikler
all_words_list = [word for tokens_list in data['tokens'] for word in tokens_list]

# Sæt af unikke ord (vocab) og beregn vocab size
vocab = set(all_words_list)

# Længden af vocab
vocab_size = len(vocab)

print(f"Antal tokens i alt (alle ord i alle artikler): {len(all_words_list)}")
print(f"Vocabulary size (unikke ord): {vocab_size}")
print(f"10 ord i the vocab (bare lige for at se): {list(vocab)[:10]}")

Antal tokens i alt (alle ord i alle artikler): 143386042
Vocabulary size (unikke ord): 858515
10 ord i the vocab (bare lige for at se): ['ipob', 'alaikaman', 'pianporn', 'wojapi', 'engagementif', 'burc', 'thucydides', 'socratics', 'holocausthowever', 'hgglunds']


In [23]:
# Liste over alle ord i alle artikler
filtered_words_list = [word for tokens_list in data['filtered_tokens'] for word in tokens_list]

# Sæt af unikke ord (vocab) og beregn vocab size
vocab_filtered = set(filtered_words_list)

# Længden af vocab
filtered_vocab_size = len(vocab_filtered)

print(f"Antal tokens i alt (alle ord i alle artikler): {len(filtered_words_list)}")
print(f"Vocabulary size (unikke ord): {filtered_vocab_size}")
print(f"10 ord i the vocab (bare lige for at se): {list(vocab_filtered)[:10]}")

Antal tokens i alt (alle ord i alle artikler): 85338872
Vocabulary size (unikke ord): 858364
10 ord i the vocab (bare lige for at se): ['ipob', 'alaikaman', 'pianporn', 'wojapi', 'engagementif', 'burc', 'thucydides', 'socratics', 'holocausthowever', 'hgglunds']


In [24]:
ps = PorterStemmer()

# Anvend stemming på de filtrerede tokens og gem det i en ny kolonne 'stemmed'
data['stemmed'] = data['filtered_tokens'].apply(
    lambda x: [ps.stem(w) if not w.startswith('<') else w for w in x]
)

# Liste over alle stemmede ord i alle artikler
all_stemmed_words = [word for sublist in data['stemmed'] for word in sublist]

# Sæt af unikke stemmede ord (vocab) og beregn vocab size
vocab_stemmed = set(all_stemmed_words)
vocab_stemmed_size = len(vocab_stemmed)

# Reduktion efter stopwords i forhold til baseline
reduction_stop = (1 - (filtered_vocab_size / vocab_size)) * 100

# Reduktion efter stemming i forhold til efter stopwords
reduction_stem = (1 - (vocab_stemmed_size / filtered_vocab_size)) * 100

# PRINT RESULTATER 
print(f"\n--- STATISTIK FOR TASK 1 ---")
print(f"Oprindelig Vocab Size: {vocab_size}")
print(f"Vocab Size efter Stopwords: {filtered_vocab_size} (Reduktion: {reduction_stop:.2f}%)")
print(f"Vocab Size efter Stemming: {vocab_stemmed_size} (Reduktion: {reduction_stem:.2f}%)")

C:\Users\bruge\AppData\Local\Temp\ipykernel_29300\1562358668.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['stemmed'] = data['filtered_tokens'].apply(



--- STATISTIK FOR TASK 1 ---
Oprindelig Vocab Size: 858515
Vocab Size efter Stopwords: 858364 (Reduktion: 0.02%)
Vocab Size efter Stemming: 718139 (Reduktion: 16.34%)


In [25]:
# Split the data (80, 10, 10)
train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)
print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")
print(f"Test size: {len(test_data)}")

Train size: 240000
Validation size: 30000
Test size: 30000


## Part 2

In [26]:
# Find unique label types (column: type)
unique_labels = data['type'].unique()
print(f"Unique label types: {unique_labels}")

# count of each label type
label_counts = data['type'].value_counts()
print("\nLabel counts:")
print(label_counts)

Unique label types: ['political' 'fake' 'satire' 'reliable' 'conspiracy' 'unreliable' 'bias'
 'rumor' 'unknown' nan 'clickbait' 'hate' 'junksci']

Label counts:
type
reliable      65768
political     59362
bias          39028
conspiracy    33623
fake          33212
rumor         16063
unknown       12477
clickbait      8793
unreliable     6092
junksci        4854
satire         3489
hate           2528
Name: count, dtype: int64


In [27]:
# Group labels (type) into reliable and unreliable
train_data['Labels'] = train_data['type'].apply(lambda x: 'news' if x == 'reliable' else 'fake news')
val_data['Labels'] = val_data['type'].apply(lambda x: 'news' if x == 'reliable' else 'fake news')

unique_labels_grouped = train_data['Labels'].unique()
print(f"\nUnique label types after grouping: {unique_labels_grouped}")
grouped_labels_counts = train_data['Labels'].value_counts()
print("\nGrouped label counts:")
print(grouped_labels_counts)


Unique label types after grouping: ['fake news' 'news']

Grouped label counts:
Labels
fake news    187389
news          52611
Name: count, dtype: int64


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

In [ ]:
train_texts = train_data['stemmed'].apply(lambda x: " ".join(x))
val_texts = val_data['stemmed'].apply(lambda x: " ".join(x))

y_train = train_data['Labels']
y_val = val_data['Labels']

vectorizer = CountVectorizer(vocabulary= vocab_stemmed, max_features=10000)

X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

   fake news       0.96      0.97      0.96     23434
        news       0.88      0.85      0.86      6566

    accuracy                           0.94     30000
   macro avg       0.92      0.91      0.91     30000
weighted avg       0.94      0.94      0.94     30000



In [35]:
# Combine titel and stemmed content into one text feature
train_combined_text = train_data['title'].fillna('') + " " + train_data['stemmed'].apply(lambda x: " ".join(x))
val_combined_text = val_data['title'].fillna('') + " " + val_data['stemmed'].apply(lambda x: " ".join(x))

X_train_text = vectorizer.fit_transform(train_combined_text)
X_val_text = vectorizer.transform(val_combined_text)

# Make meta-data feature
encoder = OneHotEncoder(handle_unknown='ignore')
X_train_domain = encoder.fit_transform(train_data[['domain']].fillna('unknown'))
X_val_domain = encoder.transform(val_data[['domain']].fillna('unknown'))

# Stack features together
X_train_final = hstack([X_train_text, X_train_domain])
X_val_final = hstack([X_val_text, X_val_domain])

meta_model = LogisticRegression(max_iter=1000, random_state=42)
meta_model.fit(X_train_final, y_train)

y_pred_meta = meta_model.predict(X_val_final)
print(classification_report(y_val, y_pred_meta))

              precision    recall  f1-score   support

   fake news       0.99      1.00      1.00     23434
        news       0.99      0.97      0.98      6566

    accuracy                           0.99     30000
   macro avg       0.99      0.99      0.99     30000
weighted avg       0.99      0.99      0.99     30000

